# PROBLEMA ENCONTRADO

Miré las 314 iteraciones. Hay tres problemas encadenados, y ninguno es "el planner anda mal": el rover queda atrapado en una geometría que tu control no puede resolver.

1. Desde el checkpoint #2, el cp#3 queda DETRÁS del rover y nunca se da vuelta

En la iteración 145 conseguís el cp#2 y el objetivo pasa a cp#3 a 28 m, rel +157 grados. A partir de ahí, rel se queda entre +100° y ±180° durante las 170 iteraciones restantes: el objetivo está literalmente a la espalda.

El problema es que todos los comandos que emitís tienen linear > 0 y angular saturado en ±0.45 (linear=+0.64 angular=-0.45 siguiendo camino (+65 grados)). El banco de 222 caminos candidatos no contiene giro en el lugar. Con un arco de radio grande, para girar 150° necesitás varios metros de espacio libre lateral — que no tenés. Además el signo oscila: en 0166 pasa de angular=-0.45 a +0.21, en 0177 aparece cambio de lado justificado (+65 grados hacia derecha). Va zigzagueando sin converger nunca.

La distancia al cp#3 baja de 29 m a 25 m en 170 iteraciones. Prácticamente no avanzó hacia ningún lado.

2. El deadlock final: la recuperación tiene el retroceso deshabilitado de hecho

Desde la iteración ~200 la pose queda congelada en (+2.22,-0.46,-34gr) hasta el final. 129 frenadas por obstáculo, desatascos forzados: 0, retrocesos: 0. El bloque de recuperación se repite idéntico 32 veces:

mapa detras del robot: libre=0% cobertura=0%
detras no parece seguro (o sin datos todavia), salteo el retroceso
rumbo +0 grados: libre=9% cobertura=42%
rumbo +180 grados: libre=0% cobertura=0%
elijo rumbo +0 grados por mapa (libre=9%)

Dos bugs acá:

La cámara nunca ve hacia atrás, así que cobertura detrás es siempre 0%. Tu condición "si no hay datos, no retrocedo" no es conservadora, es un return incondicional: el retroceso nunca se puede ejecutar en ninguna corrida. Por eso retrocesos: 0.
Con el retroceso descartado y +180 descartado por cobertura=0, la única opción que queda es +0°, o sea justo la dirección bloqueada, y la elige con libre=9%. Recupera, apunta al obstáculo, frena por obstáculo, recupera. Loop infinito. Eso es exactamente lo que ves: "se recupera y no avanza".

3. La odometría no integra el movimiento real

distancia recorrida: 1.89 m con 38 correcciones GPS y el cp#2 pasando de 40 m a alcanzado: físicamente recorrió decenas de metros. La pose se integra desde la velocidad comandada, no medida — por eso al frenar (linear=0.00) la pose se congela literalmente, y el mapa BEV nunca se recentra (recentrados: 0). Las celdas de obstáculo enfrente quedan clavadas en la misma posición del grid para siempre, aunque el robot se mueva. Eso alimenta el punto 2: el "obstáculo al frente" con clearance=0.40 m puede ser en parte un fantasma del mapa que no se puede limpiar porque el robot "no se movió".

Qué tocaría, en orden de impacto

Sacar el guard de cobertura del retroceso: si clearance < 0.5 m y el frente tiene libre < 30%, retrocedé a ciegas 0.3–0.5 m con velocidad baja y timeout. Sin eso el rover no tiene ninguna salida de un callejón.
Agregar pivote en el lugar al banco de caminos (linear=0, angular=±max) y activarlo cuando |rel| > 90°. Hoy no existe la maniobra "darse vuelta".
Integrar la pose con la velocidad reportada por el rover (o corregir con GPS/track cada N frames), y forzar decaimiento/recentrado del mapa cuando la pose no cambia pero el GPS sí.
Poner un contador de desatasco duro: si clearance no mejora en N iteraciones, ejecutar la secuencia retroceso + giro 90° sin consultar el mapa.

Si querés, abro el repo del proyecto (IROS26-LaRovernetta, rama fixVelocidad-Tom) y te marco las funciones exactas donde están el guard del retroceso y el banco de caminos.

# RUN DETALLE

```

==============================================================
  MODO REAL: el rover se va a mover. Ctrl-C frena.
  Tene el robot a la vista y espacio libre alrededor.
==============================================================
  3 ...
  2 ...
  1 ...
[bridge] Objetivo: checkpoint #1 (-21.98324013, 27.83091354)
[bridge] ✓ checkpoint #1 conseguido: {'message': 'Checkpoint reached successfully', 'next_checkpoint_sequence': 2, 'mission_completed': False}
[percep] frame 1024x576 != calibracion 1920x1080: reescalo K (sx=0.533, sy=0.533)
[0000] rumbo=283 (orientation)  meta: cp#1 a 3 m, rel +159 grados  celdas BEV=2994  mapa: 2949 celdas  pose=(+0.00,+0.00,+0gr)
[bridge] banco de 222 caminos candidatos precalculado en 0.1 s (se reusa durante toda la corrida)
  [ENVIADO] linear=+0.63 angular=-0.45  siguiendo camino (+67 grados)
[0001] rumbo=283 (orientation)  meta: cp#2 a 40 m, rel +33 grados  celdas BEV=2994  mapa: 2949 celdas  pose=(+0.00,+0.00,+0gr)
  [ENVIADO] linear=+0.82 angular=-0.25  siguiendo camino (+32 grados)
[0002] rumbo=279 (orientation)  meta: cp#2 a 40 m, rel +37 grados  celdas BEV=2994  mapa: 3235 celdas  pose=(+0.01,+0.00,+0gr)
  [ENVIADO] linear=+0.86 angular=-0.19  siguiendo camino (+24 grados)
[0003] rumbo=284 (orientation)  meta: cp#2 a 40 m, rel +32 grados  celdas BEV=2994  mapa: 4027 celdas  pose=(+0.03,+0.00,+0gr)
  [ENVIADO] linear=+0.84 angular=-0.23  siguiendo camino (+29 grados)
[0004] rumbo=284 (orientation)  meta: cp#2 a 40 m, rel +32 grados  celdas BEV=2994  mapa: 4416 celdas  pose=(+0.05,+0.00,-0gr)
  [ENVIADO] linear=+0.82 angular=-0.25  siguiendo camino (+32 grados)
[0005] rumbo=276 (orientation)  meta: cp#2 a 41 m, rel +42 grados  celdas BEV=2994  mapa: 4532 celdas  pose=(+0.07,+0.00,+0gr)
  [ENVIADO] linear=+0.80 angular=-0.29  siguiendo camino (+37 grados)
[0006] rumbo=285 (orientation)  meta: cp#2 a 41 m, rel +33 grados  celdas BEV=2994  mapa: 4645 celdas  pose=(+0.11,-0.00,-0gr)
  [ENVIADO] linear=+0.81 angular=-0.27  siguiendo camino (+34 grados)
[0007] rumbo=291 (orientation)  meta: cp#2 a 40 m, rel +29 grados  celdas BEV=2994  mapa: 4781 celdas  pose=(+0.13,-0.00,-0gr)
  [ENVIADO] linear=+0.81 angular=-0.27  siguiendo camino (+34 grados)
[0008] rumbo=291 (orientation)  meta: cp#2 a 40 m, rel +29 grados  celdas BEV=2994  mapa: 4815 celdas  pose=(+0.15,-0.00,-0gr)
  [ENVIADO] linear=+0.80 angular=-0.28  siguiendo camino (+36 grados)
[0009] rumbo=304 (orientation)  meta: cp#2 a 40 m, rel +17 grados  celdas BEV=2994  mapa: 4858 celdas  pose=(+0.16,-0.00,-0gr)
  [ENVIADO] linear=+0.83 angular=-0.24  siguiendo camino (+30 grados)
[0010] rumbo=301 (orientation)  meta: cp#2 a 40 m, rel +20 grados  celdas BEV=2994  mapa: 4950 celdas  pose=(+0.17,-0.00,+0gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0011] rumbo=330 (orientation)  meta: cp#2 a 40 m, rel -9 grados  celdas BEV=2994  mapa: 5017 celdas  pose=(+0.19,-0.00,-2gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0012] rumbo=309 (orientation)  meta: cp#2 a 40 m, rel +12 grados  celdas BEV=2994  mapa: 5005 celdas  pose=(+0.19,-0.00,-2gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=98% cobertura=77%
[bridge]   rumbo +90 grados: libre=100% cobertura=21%
[bridge]   rumbo -90 grados: libre=100% cobertura=20%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=98%)
[0013] rumbo=309 (orientation)  meta: cp#2 a 40 m, rel +12 grados  celdas BEV=2994  mapa: 4934 celdas  pose=(+0.19,-0.00,-2gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0014] rumbo=309 (orientation)  meta: cp#2 a 40 m, rel +12 grados  celdas BEV=2994  mapa: 4813 celdas  pose=(+0.19,-0.00,-2gr)
  [ENVIADO] linear=+0.93 angular=-0.10  siguiendo camino (+13 grados)
[0015] rumbo=309 (orientation)  meta: cp#2 a 40 m, rel +12 grados  celdas BEV=2994  mapa: 4846 celdas  pose=(+0.20,-0.00,-2gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0016] rumbo=320 (orientation)  meta: cp#2 a 39 m, rel +1 grados  celdas BEV=2994  mapa: 4882 celdas  pose=(+0.22,-0.00,-2gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0017] rumbo=314 (orientation)  meta: cp#2 a 39 m, rel +7 grados  celdas BEV=2994  mapa: 4733 celdas  pose=(+0.22,-0.00,-2gr)
  [ENVIADO] linear=+0.99 angular=+0.01  siguiendo camino (-1 grados)
[0018] rumbo=313 (orientation)  meta: cp#2 a 39 m, rel +9 grados  celdas BEV=2994  mapa: 4576 celdas  pose=(+0.22,-0.00,-2gr)
  [ENVIADO] linear=+0.86 angular=-0.20  siguiendo camino (+25 grados)
[0019] rumbo=305 (orientation)  meta: cp#2 a 39 m, rel +16 grados  celdas BEV=2994  mapa: 4658 celdas  pose=(+0.25,-0.00,-3gr)
  [ENVIADO] linear=+0.98 angular=-0.03  siguiendo camino (+4 grados)
[0020] rumbo=319 (orientation)  meta: cp#2 a 39 m, rel +2 grados  celdas BEV=2994  mapa: 4714 celdas  pose=(+0.26,-0.00,-3gr)
  [ENVIADO] linear=+0.99 angular=-0.01  siguiendo camino (+2 grados)
[0021] rumbo=315 (orientation)  meta: cp#2 a 39 m, rel +6 grados  celdas BEV=2994  mapa: 4796 celdas  pose=(+0.29,-0.01,-3gr)
  [ENVIADO] linear=+0.89 angular=-0.16  siguiendo camino (+20 grados)
[0022] rumbo=315 (orientation)  meta: cp#2 a 38 m, rel +7 grados  celdas BEV=2994  mapa: 4801 celdas  pose=(+0.31,-0.01,-3gr)
  [ENVIADO] linear=+0.86 angular=-0.20  siguiendo camino (+25 grados)
[0023] rumbo=329 (orientation)  meta: cp#2 a 37 m, rel -7 grados  celdas BEV=2994  mapa: 4956 celdas  pose=(+0.37,-0.01,-4gr)
  [ENVIADO] linear=+0.92 angular=+0.00  mantengo el rumbo (evito titubeo, -15 grados)
[0024] rumbo=325 (orientation)  meta: cp#2 a 37 m, rel -3 grados  celdas BEV=2994  mapa: 5064 celdas  pose=(+0.39,-0.01,-4gr)
  [ENVIADO] linear=+0.92 angular=+0.12  siguiendo camino (-15 grados)
[0025] rumbo=323 (orientation)  meta: cp#2 a 37 m, rel -1 grados  celdas BEV=2994  mapa: 5157 celdas  pose=(+0.41,-0.01,-4gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0026] rumbo=345 (orientation)  meta: cp#2 a 37 m, rel -23 grados  celdas BEV=2994  mapa: 5173 celdas  pose=(+0.42,-0.01,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0027] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -8 grados  celdas BEV=2994  mapa: 5208 celdas  pose=(+0.43,-0.01,-3gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=76% cobertura=79%
[bridge]   rumbo +90 grados: libre=99% cobertura=22%
[bridge]   rumbo -90 grados: libre=100% cobertura=23%
[bridge]   rumbo +180 grados: libre=100% cobertura=1%
[bridge]   elijo rumbo +0 grados por mapa (libre=76%)
[0028] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -8 grados  celdas BEV=2994  mapa: 5128 celdas  pose=(+0.43,-0.01,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0029] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -8 grados  celdas BEV=2994  mapa: 5119 celdas  pose=(+0.43,-0.01,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0030] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -8 grados  celdas BEV=2994  mapa: 5101 celdas  pose=(+0.43,-0.01,-3gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=76% cobertura=79%
[bridge]   rumbo +90 grados: libre=97% cobertura=21%
[bridge]   rumbo -90 grados: libre=100% cobertura=22%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=76%)
[0031] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -8 grados  celdas BEV=2994  mapa: 4914 celdas  pose=(+0.43,-0.01,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0032] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -9 grados  celdas BEV=2994  mapa: 4830 celdas  pose=(+0.43,-0.01,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0033] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -9 grados  celdas BEV=2994  mapa: 4652 celdas  pose=(+0.43,-0.01,-3gr)
  [ENVIADO] linear=+0.78 angular=-0.31  siguiendo camino (+40 grados)
[0034] rumbo=330 (orientation)  meta: cp#2 a 36 m, rel -9 grados  celdas BEV=2994  mapa: 4531 celdas  pose=(+0.43,-0.01,-3gr)
  [ENVIADO] linear=+0.78 angular=-0.31  siguiendo camino (+40 grados)
[0035] rumbo=327 (orientation)  meta: cp#2 a 36 m, rel -6 grados  celdas BEV=2994  mapa: 4490 celdas  pose=(+0.45,-0.02,-3gr)
  [ENVIADO] linear=+0.91 angular=-0.12  siguiendo camino (+16 grados)
[0036] rumbo=335 (orientation)  meta: cp#2 a 35 m, rel -13 grados  celdas BEV=2994  mapa: 4554 celdas  pose=(+0.47,-0.02,-3gr)
  [ENVIADO] linear=+0.93 angular=+0.10  siguiendo camino (-12 grados)
[0037] rumbo=337 (orientation)  meta: cp#2 a 35 m, rel -15 grados  celdas BEV=2994  mapa: 4644 celdas  pose=(+0.48,-0.02,-3gr)
  [ENVIADO] linear=+0.94 angular=+0.08  siguiendo camino (-11 grados)
[0038] rumbo=335 (orientation)  meta: cp#2 a 35 m, rel -13 grados  celdas BEV=2994  mapa: 4656 celdas  pose=(+0.50,-0.02,-3gr)
  [ENVIADO] linear=+0.94 angular=+0.09  siguiendo camino (-11 grados)
[0039] rumbo=329 (orientation)  meta: cp#2 a 35 m, rel -7 grados  celdas BEV=2994  mapa: 4693 celdas  pose=(+0.52,-0.02,-3gr)
  [ENVIADO] linear=+0.94 angular=+0.08  siguiendo camino (-10 grados)
[0040] rumbo=323 (orientation)  meta: cp#2 a 34 m, rel -1 grados  celdas BEV=2994  mapa: 4813 celdas  pose=(+0.55,-0.02,-3gr)
  [ENVIADO] linear=+0.97 angular=-0.04  siguiendo camino (+5 grados)
[0041] rumbo=329 (orientation)  meta: cp#2 a 34 m, rel -7 grados  celdas BEV=2994  mapa: 4889 celdas  pose=(+0.57,-0.02,-3gr)
  [ENVIADO] linear=+0.92 angular=+0.12  siguiendo camino (-15 grados)
[0042] rumbo=318 (orientation)  meta: cp#2 a 33 m, rel +4 grados  celdas BEV=2994  mapa: 4967 celdas  pose=(+0.59,-0.02,-3gr)
  [ENVIADO] linear=+0.97 angular=-0.04  siguiendo camino (+5 grados)
[0043] rumbo=327 (orientation)  meta: cp#2 a 33 m, rel -5 grados  celdas BEV=2994  mapa: 4959 celdas  pose=(+0.59,-0.02,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0044] rumbo=347 (orientation)  meta: cp#2 a 33 m, rel -25 grados  celdas BEV=2994  mapa: 5050 celdas  pose=(+0.61,-0.02,-3gr)
  [ENVIADO] linear=+1.00 angular=+0.01  siguiendo camino (-1 grados)
[0045] rumbo=327 (orientation)  meta: cp#2 a 33 m, rel -5 grados  celdas BEV=2994  mapa: 5042 celdas  pose=(+0.61,-0.02,-3gr)
  [ENVIADO] linear=+1.00 angular=+0.01  siguiendo camino (-1 grados)
[0046] rumbo=342 (orientation)  meta: cp#2 a 34 m, rel -20 grados  celdas BEV=2994  mapa: 5035 celdas  pose=(+0.63,-0.02,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0047] rumbo=315 (orientation)  meta: cp#2 a 33 m, rel +7 grados  celdas BEV=2994  mapa: 5065 celdas  pose=(+0.67,-0.03,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0048] rumbo=322 (orientation)  meta: cp#2 a 33 m, rel +1 grados  celdas BEV=2994  mapa: 4987 celdas  pose=(+0.67,-0.03,-3gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=63% cobertura=77%
[bridge]   rumbo +90 grados: libre=99% cobertura=20%
[bridge]   rumbo -90 grados: libre=99% cobertura=22%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=63%)
[0049] rumbo=322 (orientation)  meta: cp#2 a 33 m, rel +1 grados  celdas BEV=2994  mapa: 4984 celdas  pose=(+0.67,-0.03,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0050] rumbo=322 (orientation)  meta: cp#2 a 33 m, rel +1 grados  celdas BEV=2994  mapa: 4910 celdas  pose=(+0.67,-0.03,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0051] rumbo=322 (orientation)  meta: cp#2 a 33 m, rel +1 grados  celdas BEV=2994  mapa: 4826 celdas  pose=(+0.67,-0.03,-3gr)
  [ENVIADO] linear=+0.93 angular=-0.10  siguiendo camino (+12 grados)
[0052] rumbo=322 (orientation)  meta: cp#2 a 33 m, rel +1 grados  celdas BEV=2994  mapa: 4668 celdas  pose=(+0.67,-0.03,-3gr)
  [ENVIADO] linear=+0.93 angular=-0.10  siguiendo camino (+12 grados)
[0053] rumbo=350 (orientation)  meta: cp#2 a 33 m, rel -27 grados  celdas BEV=2994  mapa: 4626 celdas  pose=(+0.68,-0.03,-3gr)
  [ENVIADO] linear=+0.86 angular=+0.00  mantengo el rumbo (evito titubeo, -25 grados)
[0054] rumbo=318 (orientation)  meta: cp#2 a 33 m, rel +5 grados  celdas BEV=2994  mapa: 4567 celdas  pose=(+0.70,-0.03,-3gr)
  [ENVIADO] linear=+0.86 angular=+0.20  siguiendo camino (-25 grados)
[0055] rumbo=322 (orientation)  meta: cp#2 a 32 m, rel +1 grados  celdas BEV=2994  mapa: 4772 celdas  pose=(+0.73,-0.03,-4gr)
  [ENVIADO] linear=+0.87 angular=+0.00  mantengo el rumbo (evito titubeo, +23 grados)
[0056] rumbo=325 (orientation)  meta: cp#2 a 31 m, rel -3 grados  celdas BEV=2994  mapa: 4936 celdas  pose=(+0.77,-0.03,-4gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0057] rumbo=344 (orientation)  meta: cp#2 a 32 m, rel -21 grados  celdas BEV=2994  mapa: 4909 celdas  pose=(+0.78,-0.03,-4gr)
  [ENVIADO] linear=+0.90 angular=-0.14  siguiendo camino (+18 grados)
[0058] rumbo=327 (orientation)  meta: cp#2 a 31 m, rel -4 grados  celdas BEV=2994  mapa: 4905 celdas  pose=(+0.78,-0.03,-4gr)
  [ENVIADO] linear=+0.90 angular=-0.14  siguiendo camino (+18 grados)
[0059] rumbo=346 (orientation)  meta: cp#2 a 31 m, rel -23 grados  celdas BEV=2994  mapa: 4946 celdas  pose=(+0.79,-0.03,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0060] rumbo=326 (orientation)  meta: cp#2 a 31 m, rel -3 grados  celdas BEV=2994  mapa: 4887 celdas  pose=(+0.81,-0.04,-6gr)
  [ENVIADO] linear=+0.93 angular=-0.10  siguiendo camino (+12 grados)
[0061] rumbo=332 (orientation)  meta: cp#2 a 30 m, rel -9 grados  celdas BEV=2994  mapa: 4943 celdas  pose=(+0.82,-0.04,-6gr)
  [ENVIADO] linear=+0.94 angular=-0.08  siguiendo camino (+10 grados)
[0062] rumbo=328 (orientation)  meta: cp#2 a 30 m, rel -5 grados  celdas BEV=2994  mapa: 5046 celdas  pose=(+0.84,-0.04,-6gr)
  [ENVIADO] linear=+0.96 angular=-0.06  siguiendo camino (+8 grados)
[0063] rumbo=322 (orientation)  meta: cp#2 a 29 m, rel +2 grados  celdas BEV=2994  mapa: 5168 celdas  pose=(+0.87,-0.04,-5gr)
  [ENVIADO] linear=+0.95 angular=-0.06  siguiendo camino (+8 grados)
[0064] rumbo=349 (orientation)  meta: cp#2 a 29 m, rel -25 grados  celdas BEV=2994  mapa: 5220 celdas  pose=(+0.89,-0.04,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0065] rumbo=353 (orientation)  meta: cp#2 a 29 m, rel -29 grados  celdas BEV=2994  mapa: 5182 celdas  pose=(+0.89,-0.04,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0066] rumbo=330 (orientation)  meta: cp#2 a 29 m, rel -6 grados  celdas BEV=2994  mapa: 5126 celdas  pose=(+0.89,-0.04,-6gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=69% cobertura=78%
[bridge]   rumbo +90 grados: libre=61% cobertura=22%
[bridge]   rumbo -90 grados: libre=100% cobertura=20%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=69%)
[0067] rumbo=331 (orientation)  meta: cp#2 a 29 m, rel -8 grados  celdas BEV=2994  mapa: 5007 celdas  pose=(+0.89,-0.04,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0068] rumbo=331 (orientation)  meta: cp#2 a 29 m, rel -9 grados  celdas BEV=2994  mapa: 4899 celdas  pose=(+0.90,-0.05,-6gr)
  [ENVIADO] linear=+1.00 angular=-0.00  siguiendo camino (+0 grados)
[0069] rumbo=331 (orientation)  meta: cp#2 a 28 m, rel -10 grados  celdas BEV=2994  mapa: 4849 celdas  pose=(+0.90,-0.05,-6gr)
  [ENVIADO] linear=+0.99 angular=+0.01  siguiendo camino (-1 grados)
[0070] rumbo=326 (orientation)  meta: cp#2 a 28 m, rel -5 grados  celdas BEV=2994  mapa: 4908 celdas  pose=(+0.92,-0.05,-6gr)
  [ENVIADO] linear=+0.99 angular=+0.01  siguiendo camino (-1 grados)
[0071] rumbo=349 (orientation)  meta: cp#2 a 28 m, rel -29 grados  celdas BEV=2994  mapa: 4868 celdas  pose=(+0.94,-0.05,-3gr)
  [ENVIADO] linear=+0.98 angular=-0.03  siguiendo camino (+4 grados)
[0072] rumbo=324 (orientation)  meta: cp#2 a 27 m, rel -3 grados  celdas BEV=2994  mapa: 4871 celdas  pose=(+0.96,-0.05,-3gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0073] rumbo=349 (orientation)  meta: cp#2 a 27 m, rel -29 grados  celdas BEV=2994  mapa: 4943 celdas  pose=(+0.97,-0.05,-1gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0074] rumbo=326 (orientation)  meta: cp#2 a 27 m, rel -5 grados  celdas BEV=2994  mapa: 4876 celdas  pose=(+0.97,-0.05,-1gr)
  [ENVIADO] linear=+0.78 angular=-0.32  siguiendo camino (+40 grados)
[0075] rumbo=326 (orientation)  meta: cp#2 a 27 m, rel -5 grados  celdas BEV=2994  mapa: 4781 celdas  pose=(+0.97,-0.05,-1gr)
  [ENVIADO] linear=+0.78 angular=-0.32  siguiendo camino (+40 grados)
[0076] rumbo=328 (orientation)  meta: cp#2 a 27 m, rel -7 grados  celdas BEV=2994  mapa: 4835 celdas  pose=(+0.99,-0.05,-1gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0077] rumbo=352 (orientation)  meta: cp#2 a 27 m, rel -30 grados  celdas BEV=2994  mapa: 4885 celdas  pose=(+1.00,-0.05,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0078] rumbo=331 (orientation)  meta: cp#2 a 27 m, rel -8 grados  celdas BEV=2994  mapa: 4925 celdas  pose=(+1.02,-0.05,-6gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=69% cobertura=75%
[bridge]   rumbo +90 grados: libre=43% cobertura=20%
[bridge]   rumbo -90 grados: libre=100% cobertura=20%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=69%)
[0079] rumbo=331 (orientation)  meta: cp#2 a 27 m, rel -9 grados  celdas BEV=2994  mapa: 4903 celdas  pose=(+1.02,-0.05,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0080] rumbo=331 (orientation)  meta: cp#2 a 27 m, rel -9 grados  celdas BEV=2994  mapa: 4887 celdas  pose=(+1.02,-0.05,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0081] rumbo=330 (orientation)  meta: cp#2 a 27 m, rel -9 grados  celdas BEV=2994  mapa: 4838 celdas  pose=(+1.02,-0.05,-6gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=72% cobertura=74%
[bridge]   rumbo +90 grados: libre=47% cobertura=19%
[bridge]   rumbo -90 grados: libre=100% cobertura=20%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=72%)
[0082] rumbo=331 (orientation)  meta: cp#2 a 27 m, rel -10 grados  celdas BEV=2994  mapa: 4812 celdas  pose=(+1.02,-0.05,-6gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0083] rumbo=331 (orientation)  meta: cp#2 a 27 m, rel -11 grados  celdas BEV=2994  mapa: 4694 celdas  pose=(+1.02,-0.05,-6gr)
  [ENVIADO] linear=+0.83 angular=-0.24  siguiendo camino (+31 grados)
[0084] rumbo=331 (orientation)  meta: cp#2 a 27 m, rel -11 grados  celdas BEV=2994  mapa: 4558 celdas  pose=(+1.02,-0.05,-6gr)
  [ENVIADO] linear=+0.83 angular=-0.24  siguiendo camino (+31 grados)
[0085] rumbo=348 (orientation)  meta: cp#2 a 27 m, rel -28 grados  celdas BEV=2994  mapa: 4639 celdas  pose=(+1.03,-0.05,-7gr)
  [ENVIADO] linear=+0.86 angular=-0.19  siguiendo camino (+25 grados)
[0086] rumbo=333 (orientation)  meta: cp#2 a 27 m, rel -14 grados  celdas BEV=2994  mapa: 4704 celdas  pose=(+1.05,-0.06,-7gr)
  [ENVIADO] linear=+0.86 angular=-0.20  siguiendo camino (+25 grados)
[0087] rumbo=356 (orientation)  meta: cp#2 a 26 m, rel -38 grados  celdas BEV=2994  mapa: 4836 celdas  pose=(+1.06,-0.06,-8gr)
  [ENVIADO] linear=+0.96 angular=+0.05  siguiendo camino (-7 grados)
[0088] rumbo=2 (orientation)  meta: cp#2 a 26 m, rel -44 grados  celdas BEV=2994  mapa: 5076 celdas  pose=(+1.12,-0.07,-12gr)
  [ENVIADO] linear=+0.94 angular=+0.09  siguiendo camino (-11 grados)
[0089] rumbo=2 (orientation)  meta: cp#2 a 26 m, rel -45 grados  celdas BEV=2994  mapa: 5140 celdas  pose=(+1.14,-0.07,-12gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0090] rumbo=12 (orientation)  meta: cp#2 a 26 m, rel -56 grados  celdas BEV=2994  mapa: 5040 celdas  pose=(+1.16,-0.08,-12gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0091] rumbo=356 (orientation)  meta: cp#2 a 25 m, rel -41 grados  celdas BEV=2994  mapa: 5069 celdas  pose=(+1.17,-0.08,-12gr)
  [ENVIADO] linear=+0.78 angular=+0.31  siguiendo camino (-40 grados)
[0092] rumbo=356 (orientation)  meta: cp#2 a 25 m, rel -41 grados  celdas BEV=2994  mapa: 5069 celdas  pose=(+1.17,-0.08,-12gr)
  [ENVIADO] linear=+0.84 angular=+0.22  siguiendo camino (-29 grados)
[0093] rumbo=350 (orientation)  meta: cp#2 a 25 m, rel -36 grados  celdas BEV=2994  mapa: 5196 celdas  pose=(+1.19,-0.08,-13gr)
  [ENVIADO] linear=+0.84 angular=+0.23  siguiendo camino (-29 grados)
[0094] rumbo=341 (orientation)  meta: cp#2 a 25 m, rel -28 grados  celdas BEV=2994  mapa: 5270 celdas  pose=(+1.20,-0.09,-12gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0095] rumbo=357 (orientation)  meta: cp#2 a 25 m, rel -44 grados  celdas BEV=2994  mapa: 5459 celdas  pose=(+1.22,-0.09,-18gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0096] rumbo=345 (orientation)  meta: cp#2 a 25 m, rel -32 grados  celdas BEV=2994  mapa: 5496 celdas  pose=(+1.23,-0.09,-18gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=86% cobertura=80%
[bridge]   rumbo +90 grados: libre=100% cobertura=31%
[bridge]   rumbo -90 grados: libre=96% cobertura=18%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +90 grados por mapa (libre=100%)
  [ENVIADO] linear=+0.00 angular=-0.45  girando hacia +90 grados (regimen cercano)
[bridge]   frente libre por mapa tras girar +45 grados, corto
  [ENVIADO] linear=+0.00 angular=+0.00  fin del giro
[0097] rumbo=344 (orientation)  meta: cp#2 a 25 m, rel -30 grados  celdas BEV=2994  mapa: 5308 celdas  pose=(+1.23,-0.09,-18gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0098] rumbo=349 (orientation)  meta: cp#2 a 25 m, rel -34 grados  celdas BEV=2994  mapa: 5267 celdas  pose=(+1.23,-0.09,-18gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0099] rumbo=349 (orientation)  meta: cp#2 a 25 m, rel -32 grados  celdas BEV=2994  mapa: 5137 celdas  pose=(+1.23,-0.09,-18gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=78% cobertura=78%
[bridge]   rumbo +90 grados: libre=100% cobertura=26%
[bridge]   rumbo -90 grados: libre=94% cobertura=18%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +90 grados por mapa (libre=100%)
  [ENVIADO] linear=+0.00 angular=-0.45  girando hacia +90 grados (regimen cercano)
[bridge]   frente libre por mapa tras girar +45 grados, corto
  [ENVIADO] linear=+0.00 angular=+0.00  fin del giro
[0100] rumbo=349 (orientation)  meta: cp#2 a 25 m, rel -32 grados  celdas BEV=2994  mapa: 4827 celdas  pose=(+1.23,-0.09,-18gr)
  [ENVIADO] linear=+0.79 angular=+0.30  siguiendo camino (-38 grados)
[0101] rumbo=350 (orientation)  meta: cp#2 a 25 m, rel -32 grados  celdas BEV=2994  mapa: 4624 celdas  pose=(+1.23,-0.09,-18gr)
  [ENVIADO] linear=+0.79 angular=+0.30  siguiendo camino (-38 grados)
[0102] rumbo=13 (orientation)  meta: cp#2 a 25 m, rel -54 grados  celdas BEV=2994  mapa: 4647 celdas  pose=(+1.23,-0.09,-19gr)
  [ENVIADO] linear=+0.82 angular=+0.25  siguiendo camino (-32 grados)
[0103] rumbo=345 (orientation)  meta: cp#2 a 25 m, rel -26 grados  celdas BEV=2994  mapa: 4793 celdas  pose=(+1.25,-0.10,-19gr)
  [ENVIADO] linear=+0.78 angular=+0.31  siguiendo camino (-40 grados)
[0104] rumbo=341 (orientation)  meta: cp#2 a 25 m, rel -22 grados  celdas BEV=2994  mapa: 4795 celdas  pose=(+1.27,-0.11,-19gr)
  [ENVIADO] linear=+0.77 angular=+0.33  siguiendo camino (-42 grados)
[0105] rumbo=334 (orientation)  meta: cp#2 a 25 m, rel -15 grados  celdas BEV=2994  mapa: 4816 celdas  pose=(+1.28,-0.11,-19gr)
  [ENVIADO] linear=+0.76 angular=+0.33  siguiendo camino (-43 grados)
[0106] rumbo=329 (orientation)  meta: cp#2 a 25 m, rel -10 grados  celdas BEV=2994  mapa: 4687 celdas  pose=(+1.30,-0.12,-19gr)
  [ENVIADO] linear=+0.81 angular=+0.27  siguiendo camino (-34 grados)
[0107] rumbo=316 (orientation)  meta: cp#2 a 25 m, rel +3 grados  celdas BEV=2994  mapa: 4733 celdas  pose=(+1.32,-0.12,-19gr)
  [ENVIADO] linear=+0.97 angular=-0.05  siguiendo camino (+6 grados)
[0108] rumbo=320 (orientation)  meta: cp#2 a 25 m, rel -1 grados  celdas BEV=2994  mapa: 4752 celdas  pose=(+1.32,-0.12,-19gr)
  [ENVIADO] linear=+0.98 angular=-0.03  siguiendo camino (+4 grados)
[0109] rumbo=318 (orientation)  meta: cp#2 a 25 m, rel +1 grados  celdas BEV=2994  mapa: 4808 celdas  pose=(+1.34,-0.13,-18gr)
  [ENVIADO] linear=+0.92 angular=+0.11  siguiendo camino (-14 grados)
[0110] rumbo=322 (orientation)  meta: cp#2 a 24 m, rel -2 grados  celdas BEV=2994  mapa: 4939 celdas  pose=(+1.39,-0.15,-18gr)
  [ENVIADO] linear=+0.89 angular=+0.16  siguiendo camino (-20 grados)
[0111] rumbo=327 (orientation)  meta: cp#2 a 24 m, rel -8 grados  celdas BEV=2994  mapa: 5081 celdas  pose=(+1.44,-0.16,-14gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0112] rumbo=311 (orientation)  meta: cp#2 a 23 m, rel +8 grados  celdas BEV=2994  mapa: 5259 celdas  pose=(+1.48,-0.17,-15gr)
  [ENVIADO] linear=+0.95 angular=+0.07  siguiendo camino (-8 grados)
[0113] rumbo=314 (orientation)  meta: cp#2 a 23 m, rel +5 grados  celdas BEV=2994  mapa: 5204 celdas  pose=(+1.48,-0.17,-15gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0114] rumbo=311 (orientation)  meta: cp#2 a 23 m, rel +8 grados  celdas BEV=2994  mapa: 5264 celdas  pose=(+1.50,-0.18,-15gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0115] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +5 grados  celdas BEV=2994  mapa: 5233 celdas  pose=(+1.51,-0.18,-15gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=90% cobertura=79%
[bridge]   rumbo +90 grados: libre=100% cobertura=20%
[bridge]   rumbo -90 grados: libre=95% cobertura=26%
[bridge]   rumbo +180 grados: libre=100% cobertura=1%
[bridge]   elijo rumbo -90 grados por mapa (libre=95%)
  [ENVIADO] linear=+0.00 angular=+0.45  girando hacia -90 grados (regimen cercano)
[bridge]   frente libre por mapa tras girar -45 grados, corto
  [ENVIADO] linear=+0.00 angular=+0.00  fin del giro
[0116] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +5 grados  celdas BEV=2994  mapa: 5163 celdas  pose=(+1.51,-0.18,-15gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0117] rumbo=316 (orientation)  meta: cp#2 a 22 m, rel +3 grados  celdas BEV=2994  mapa: 5048 celdas  pose=(+1.51,-0.18,-15gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0118] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +3 grados  celdas BEV=2994  mapa: 4868 celdas  pose=(+1.51,-0.18,-15gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=95% cobertura=75%
[bridge]   rumbo +90 grados: libre=100% cobertura=19%
[bridge]   rumbo -90 grados: libre=98% cobertura=24%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=95%)
[0119] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +2 grados  celdas BEV=2994  mapa: 4834 celdas  pose=(+1.51,-0.18,-15gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0120] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +1 grados  celdas BEV=2994  mapa: 4714 celdas  pose=(+1.51,-0.18,-15gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0121] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +1 grados  celdas BEV=2994  mapa: 4639 celdas  pose=(+1.51,-0.18,-15gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=97% cobertura=73%
[bridge]   rumbo +90 grados: libre=100% cobertura=18%
[bridge]   rumbo -90 grados: libre=98% cobertura=23%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=97%)
[0122] rumbo=315 (orientation)  meta: cp#2 a 22 m, rel +1 grados  celdas BEV=2994  mapa: 4394 celdas  pose=(+1.51,-0.18,-15gr)
  [ENVIADO] linear=+0.86 angular=+0.20  siguiendo camino (-25 grados)
[0123] rumbo=317 (orientation)  meta: cp#2 a 22 m, rel -1 grados  celdas BEV=2994  mapa: 4091 celdas  pose=(+1.51,-0.18,-15gr)
  [ENVIADO] linear=+0.86 angular=+0.20  siguiendo camino (-25 grados)
[0124] rumbo=339 (orientation)  meta: cp#2 a 21 m, rel -23 grados  celdas BEV=2994  mapa: 4173 celdas  pose=(+1.51,-0.18,-14gr)
  [ENVIADO] linear=+0.90 angular=+0.14  siguiendo camino (-17 grados)
[0125] rumbo=306 (orientation)  meta: cp#2 a 21 m, rel +10 grados  celdas BEV=2994  mapa: 4348 celdas  pose=(+1.53,-0.19,-14gr)
  [ENVIADO] linear=+0.94 angular=-0.09  siguiendo camino (+11 grados)
[0126] rumbo=310 (orientation)  meta: cp#2 a 21 m, rel +7 grados  celdas BEV=2994  mapa: 4482 celdas  pose=(+1.55,-0.19,-14gr)
  [ENVIADO] linear=+0.94 angular=-0.09  siguiendo camino (+11 grados)
[0127] rumbo=305 (orientation)  meta: cp#2 a 21 m, rel +13 grados  celdas BEV=2994  mapa: 4621 celdas  pose=(+1.57,-0.20,-14gr)
  [ENVIADO] linear=+0.92 angular=-0.12  siguiendo camino (+15 grados)
[0128] rumbo=310 (orientation)  meta: cp#2 a 20 m, rel +9 grados  celdas BEV=2994  mapa: 4737 celdas  pose=(+1.60,-0.20,-15gr)
  [ENVIADO] linear=+0.91 angular=-0.12  siguiendo camino (+16 grados)
[0129] rumbo=315 (orientation)  meta: cp#2 a 19 m, rel +5 grados  celdas BEV=2994  mapa: 4845 celdas  pose=(+1.62,-0.21,-16gr)
  [ENVIADO] linear=+0.97 angular=+0.05  siguiendo camino (-6 grados)
[0130] rumbo=320 (orientation)  meta: cp#2 a 19 m, rel +1 grados  celdas BEV=2994  mapa: 4971 celdas  pose=(+1.65,-0.22,-16gr)
  [ENVIADO] linear=+0.97 angular=+0.05  siguiendo camino (-6 grados)
[0131] rumbo=316 (orientation)  meta: cp#2 a 18 m, rel +6 grados  celdas BEV=2994  mapa: 5087 celdas  pose=(+1.68,-0.22,-16gr)
  [ENVIADO] linear=+0.97 angular=+0.05  siguiendo camino (-6 grados)
[0132] rumbo=313 (orientation)  meta: cp#2 a 18 m, rel +9 grados  celdas BEV=2994  mapa: 5170 celdas  pose=(+1.70,-0.23,-16gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0133] rumbo=321 (orientation)  meta: cp#2 a 17 m, rel +2 grados  celdas BEV=2994  mapa: 5217 celdas  pose=(+1.72,-0.24,-17gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0134] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel +4 grados  celdas BEV=2994  mapa: 5251 celdas  pose=(+1.73,-0.24,-17gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=73% cobertura=80%
[bridge]   rumbo +90 grados: libre=90% cobertura=23%
[bridge]   rumbo -90 grados: libre=89% cobertura=22%
[bridge]   rumbo +180 grados: libre=100% cobertura=1%
[bridge]   elijo rumbo +0 grados por mapa (libre=73%)
[0135] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel +3 grados  celdas BEV=2994  mapa: 5160 celdas  pose=(+1.73,-0.24,-17gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0136] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel +1 grados  celdas BEV=2994  mapa: 5124 celdas  pose=(+1.73,-0.24,-17gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0137] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel -0 grados  celdas BEV=2994  mapa: 5103 celdas  pose=(+1.73,-0.24,-17gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=74% cobertura=79%
[bridge]   rumbo +90 grados: libre=94% cobertura=23%
[bridge]   rumbo -90 grados: libre=86% cobertura=20%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=74%)
[0138] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel -0 grados  celdas BEV=2994  mapa: 5003 celdas  pose=(+1.73,-0.24,-17gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0139] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel -1 grados  celdas BEV=2994  mapa: 4850 celdas  pose=(+1.73,-0.24,-17gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0140] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel -1 grados  celdas BEV=2994  mapa: 4674 celdas  pose=(+1.73,-0.24,-17gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=76% cobertura=73%
[bridge]   rumbo +90 grados: libre=96% cobertura=20%
[bridge]   rumbo -90 grados: libre=85% cobertura=19%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=76%)
[0141] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel -1 grados  celdas BEV=2994  mapa: 4516 celdas  pose=(+1.73,-0.24,-17gr)
  [ENVIADO] linear=+0.78 angular=-0.31  siguiendo camino (+40 grados)
[0142] rumbo=320 (orientation)  meta: cp#2 a 16 m, rel -2 grados  celdas BEV=2994  mapa: 4331 celdas  pose=(+1.73,-0.24,-17gr)
  [ENVIADO] linear=+0.78 angular=-0.31  siguiendo camino (+40 grados)
[0143] rumbo=318 (orientation)  meta: cp#2 a 16 m, rel -1 grados  celdas BEV=2994  mapa: 4390 celdas  pose=(+1.75,-0.25,-17gr)
  [ENVIADO] linear=+0.93 angular=-0.10  siguiendo camino (+13 grados)
[bridge] ✓ checkpoint #2 conseguido: {'message': 'Checkpoint reached successfully', 'next_checkpoint_sequence': 3, 'mission_completed': False}
[0144] rumbo=323 (orientation)  meta: cp#2 a 15 m, rel -5 grados  celdas BEV=2994  mapa: 4510 celdas  pose=(+1.77,-0.25,-17gr)
  [ENVIADO] linear=+0.96 angular=-0.06  siguiendo camino (+8 grados)
[0145] rumbo=335 (orientation)  meta: cp#3 a 28 m, rel +157 grados  celdas BEV=2994  mapa: 4352 celdas  pose=(+1.77,-0.25,-17gr)
  [ENVIADO] linear=+0.65 angular=-0.45  siguiendo camino (+63 grados)
[0146] rumbo=324 (orientation)  meta: cp#3 a 28 m, rel +168 grados  celdas BEV=2994  mapa: 4552 celdas  pose=(+1.79,-0.26,-17gr)
  [ENVIADO] linear=+0.67 angular=-0.45  siguiendo camino (+59 grados)
[0147] rumbo=335 (orientation)  meta: cp#3 a 29 m, rel +157 grados  celdas BEV=2994  mapa: 4665 celdas  pose=(+1.83,-0.27,-16gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+62 grados)
[0148] rumbo=344 (orientation)  meta: cp#3 a 29 m, rel +148 grados  celdas BEV=2994  mapa: 4711 celdas  pose=(+1.84,-0.27,-16gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+62 grados)
[0149] rumbo=349 (orientation)  meta: cp#3 a 29 m, rel +144 grados  celdas BEV=2994  mapa: 4758 celdas  pose=(+1.85,-0.28,-16gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+62 grados)
[0150] rumbo=357 (orientation)  meta: cp#3 a 29 m, rel +135 grados  celdas BEV=2994  mapa: 4793 celdas  pose=(+1.87,-0.28,-16gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+62 grados)
[0151] rumbo=6 (orientation)  meta: cp#3 a 29 m, rel +126 grados  celdas BEV=2994  mapa: 4827 celdas  pose=(+1.88,-0.28,-16gr)
  [ENVIADO] linear=+0.67 angular=-0.45  siguiendo camino (+60 grados)
[0152] rumbo=8 (orientation)  meta: cp#3 a 29 m, rel +124 grados  celdas BEV=2994  mapa: 4860 celdas  pose=(+1.89,-0.29,-16gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0153] rumbo=22 (orientation)  meta: cp#3 a 29 m, rel +110 grados  celdas BEV=2994  mapa: 4887 celdas  pose=(+1.90,-0.29,-19gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0154] rumbo=6 (orientation)  meta: cp#3 a 29 m, rel +127 grados  celdas BEV=2994  mapa: 4772 celdas  pose=(+1.90,-0.29,-19gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=91% cobertura=75%
[bridge]   rumbo +90 grados: libre=100% cobertura=21%
[bridge]   rumbo -90 grados: libre=99% cobertura=19%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=91%)
[0155] rumbo=5 (orientation)  meta: cp#3 a 29 m, rel +128 grados  celdas BEV=2994  mapa: 4700 celdas  pose=(+1.90,-0.29,-19gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+62 grados)
[0156] rumbo=6 (orientation)  meta: cp#3 a 29 m, rel +127 grados  celdas BEV=2994  mapa: 4694 celdas  pose=(+1.90,-0.29,-19gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0157] rumbo=21 (orientation)  meta: cp#3 a 29 m, rel +112 grados  celdas BEV=2994  mapa: 4657 celdas  pose=(+1.91,-0.30,-19gr)
  [ENVIADO] linear=+0.65 angular=-0.45  siguiendo camino (+63 grados)
[0158] rumbo=10 (orientation)  meta: cp#3 a 28 m, rel +123 grados  celdas BEV=2994  mapa: 4560 celdas  pose=(+1.91,-0.30,-19gr)
  [ENVIADO] linear=+0.65 angular=-0.45  siguiendo camino (+63 grados)
[0159] rumbo=23 (orientation)  meta: cp#3 a 28 m, rel +110 grados  celdas BEV=2994  mapa: 4599 celdas  pose=(+1.92,-0.30,-21gr)
  [ENVIADO] linear=+0.65 angular=-0.45  siguiendo camino (+63 grados)
[0160] rumbo=19 (orientation)  meta: cp#3 a 28 m, rel +115 grados  celdas BEV=2994  mapa: 4636 celdas  pose=(+1.96,-0.31,-21gr)
  [ENVIADO] linear=+0.64 angular=-0.45  siguiendo camino (+64 grados)
[0161] rumbo=34 (orientation)  meta: cp#3 a 28 m, rel +101 grados  celdas BEV=2994  mapa: 4673 celdas  pose=(+1.97,-0.31,-21gr)
  [ENVIADO] linear=+0.64 angular=-0.45  siguiendo camino (+65 grados)
[0162] rumbo=53 (orientation)  meta: cp#3 a 27 m, rel +81 grados  celdas BEV=2994  mapa: 4697 celdas  pose=(+1.98,-0.32,-23gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+60 grados)
[0163] rumbo=34 (orientation)  meta: cp#3 a 27 m, rel +100 grados  celdas BEV=2994  mapa: 4790 celdas  pose=(+1.99,-0.32,-27gr)
  [ENVIADO] linear=+0.68 angular=-0.45  siguiendo camino (+57 grados)
[0164] rumbo=29 (orientation)  meta: cp#3 a 27 m, rel +105 grados  celdas BEV=2994  mapa: 4819 celdas  pose=(+2.00,-0.33,-28gr)
  [ENVIADO] linear=+0.66 angular=-0.45  siguiendo camino (+62 grados)
[0165] rumbo=15 (orientation)  meta: cp#3 a 27 m, rel +119 grados  celdas BEV=2994  mapa: 4729 celdas  pose=(+2.01,-0.34,-29gr)
  [ENVIADO] linear=+0.67 angular=-0.45  siguiendo camino (+60 grados)
[0166] rumbo=304 (orientation)  meta: cp#3 a 27 m, rel -171 grados  celdas BEV=2994  mapa: 4758 celdas  pose=(+2.02,-0.34,-29gr)
  [ENVIADO] linear=+0.85 angular=+0.21  siguiendo camino (-27 grados)
[0167] rumbo=281 (orientation)  meta: cp#3 a 27 m, rel -148 grados  celdas BEV=2994  mapa: 4786 celdas  pose=(+2.04,-0.35,-30gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0168] rumbo=285 (orientation)  meta: cp#3 a 27 m, rel -152 grados  celdas BEV=2994  mapa: 4872 celdas  pose=(+2.05,-0.36,-29gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0169] rumbo=285 (orientation)  meta: cp#3 a 27 m, rel -152 grados  celdas BEV=2994  mapa: 4800 celdas  pose=(+2.05,-0.36,-29gr)
[bridge] RECUPERACION: buscando rumbo transitable (3 planes vacios seguidos)
[bridge]   rumbo +0 grados: libre=73% cobertura=76%
[bridge]   rumbo +90 grados: libre=88% cobertura=27%
[bridge]   rumbo -90 grados: libre=92% cobertura=18%
[bridge]   rumbo +180 grados: libre=100% cobertura=0%
[bridge]   elijo rumbo +90 grados por mapa (libre=88%)
  [ENVIADO] linear=+0.00 angular=-0.45  girando hacia +90 grados (regimen cercano)
[bridge]   frente libre por mapa tras girar +45 grados, corto
  [ENVIADO] linear=+0.00 angular=+0.00  fin del giro
[0170] rumbo=285 (orientation)  meta: cp#3 a 27 m, rel -152 grados  celdas BEV=2994  mapa: 4655 celdas  pose=(+2.05,-0.36,-29gr)
  [ENVIADO] linear=+0.00 angular=+0.00  el planner no encontro camino
[0171] rumbo=285 (orientation)  meta: cp#3 a 27 m, rel -152 grados  celdas BEV=2994  mapa: 4555 celdas  pose=(+2.05,-0.36,-29gr)
  [ENVIADO] linear=+0.77 angular=+0.32  siguiendo camino (-41 grados)
[0172] rumbo=285 (orientation)  meta: cp#3 a 27 m, rel -152 grados  celdas BEV=2994  mapa: 4501 celdas  pose=(+2.05,-0.36,-29gr)
  [ENVIADO] linear=+0.80 angular=+0.28  siguiendo camino (-36 grados)
[0173] rumbo=295 (orientation)  meta: cp#3 a 27 m, rel -162 grados  celdas BEV=2994  mapa: 4406 celdas  pose=(+2.06,-0.36,-29gr)
  [ENVIADO] linear=+0.80 angular=+0.28  siguiendo camino (-36 grados)
[0174] rumbo=282 (orientation)  meta: cp#3 a 27 m, rel -150 grados  celdas BEV=2994  mapa: 4319 celdas  pose=(+2.10,-0.38,-29gr)
  [ENVIADO] linear=+0.70 angular=+0.42  siguiendo camino (-54 grados)
[0175] rumbo=268 (orientation)  meta: cp#3 a 26 m, rel -137 grados  celdas BEV=2994  mapa: 4289 celdas  pose=(+2.11,-0.39,-29gr)
  [ENVIADO] linear=+0.67 angular=+0.45  siguiendo camino (-60 grados)
[0176] rumbo=270 (orientation)  meta: cp#3 a 26 m, rel -140 grados  celdas BEV=2994  mapa: 4283 celdas  pose=(+2.13,-0.40,-29gr)
  [ENVIADO] linear=+0.67 angular=+0.45  siguiendo camino (-60 grados)
[0177] rumbo=309 (orientation)  meta: cp#3 a 26 m, rel +180 grados  celdas BEV=2994  mapa: 4317 celdas  pose=(+2.13,-0.40,-25gr)
[bridge] cambio de lado justificado (+65 grados hacia derecha)
  [ENVIADO] linear=+0.64 angular=-0.45  siguiendo camino (+65 grados)
[0178] rumbo=346 (orientation)  meta: cp#3 a 26 m, rel +143 grados  celdas BEV=2994  mapa: 4313 celdas  pose=(+2.14,-0.41,-25gr)
  [ENVIADO] linear=+0.63 angular=-0.45  siguiendo camino (+66 grados)
[0179] rumbo=5 (orientation)  meta: cp#3 a 26 m, rel +123 grados  celdas BEV=2994  mapa: 4311 celdas  pose=(+2.15,-0.41,-29gr)
  [ENVIADO] linear=+0.62 angular=-0.45  siguiendo camino (+69 grados)
[0180] rumbo=10 (orientation)  meta: cp#3 a 26 m, rel +118 grados  celdas BEV=2994  mapa: 4355 celdas  pose=(+2.16,-0.42,-32gr)
  [ENVIADO] linear=+0.63 angular=-0.45  siguiendo camino (+66 grados)
[0181] rumbo=344 (orientation)  meta: cp#3 a 25 m, rel +142 grados  celdas BEV=2994  mapa: 4381 celdas  pose=(+2.18,-0.43,-33gr)
  [ENVIADO] linear=+0.60 angular=-0.45  siguiendo camino (+71 grados)
[0182] rumbo=294 (orientation)  meta: cp#3 a 25 m, rel -168 grados  celdas BEV=2994  mapa: 4396 celdas  pose=(+2.19,-0.44,-33gr)
  [ENVIADO] linear=+0.60 angular=-0.45  siguiendo camino (+72 grados)
[0183] rumbo=283 (orientation)  meta: cp#3 a 25 m, rel -157 grados  celdas BEV=2994  mapa: 4415 celdas  pose=(+2.20,-0.44,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0184] rumbo=289 (orientation)  meta: cp#3 a 25 m, rel -165 grados  celdas BEV=2994  mapa: 4428 celdas  pose=(+2.21,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0185] rumbo=289 (orientation)  meta: cp#3 a 25 m, rel -164 grados  celdas BEV=2994  mapa: 4390 celdas  pose=(+2.21,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0186] rumbo=289 (orientation)  meta: cp#3 a 26 m, rel -164 grados  celdas BEV=2994  mapa: 4282 celdas  pose=(+2.21,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=19% cobertura=70%
[bridge]   rumbo +90 grados: libre=47% cobertura=22%
[bridge]   rumbo -90 grados: libre=13% cobertura=18%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=19%)
[0187] rumbo=289 (orientation)  meta: cp#3 a 26 m, rel -164 grados  celdas BEV=2994  mapa: 4265 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0188] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -167 grados  celdas BEV=2994  mapa: 4228 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0189] rumbo=289 (orientation)  meta: cp#3 a 26 m, rel -166 grados  celdas BEV=2994  mapa: 4171 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0190] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -168 grados  celdas BEV=2994  mapa: 3998 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=22% cobertura=67%
[bridge]   rumbo +90 grados: libre=30% cobertura=19%
[bridge]   rumbo -90 grados: libre=12% cobertura=18%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=22%)
[0191] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -167 grados  celdas BEV=2994  mapa: 3820 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0192] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 3678 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0193] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -169 grados  celdas BEV=2994  mapa: 3507 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0194] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -169 grados  celdas BEV=2994  mapa: 3237 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=17% cobertura=56%
[bridge]   rumbo +90 grados: libre=22% cobertura=16%
[bridge]   rumbo -90 grados: libre=10% cobertura=17%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=17%)
[0195] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -169 grados  celdas BEV=2994  mapa: 2887 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0196] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -169 grados  celdas BEV=2994  mapa: 2887 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0197] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -169 grados  celdas BEV=2994  mapa: 2887 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0198] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=2% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0199] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0200] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0201] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0202] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0203] rumbo=289 (orientation)  meta: cp#3 a 28 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0204] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0205] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0206] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=11% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0207] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0208] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0209] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0210] rumbo=290 (orientation)  meta: cp#3 a 28 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0211] rumbo=290 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0212] rumbo=290 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0213] rumbo=290 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0214] rumbo=290 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0215] rumbo=290 (orientation)  meta: cp#3 a 27 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0216] rumbo=289 (orientation)  meta: cp#3 a 27 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0217] rumbo=290 (orientation)  meta: cp#3 a 27 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0218] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=3% cobertura=43%
[bridge]   rumbo +90 grados: libre=9% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=3%)
[0219] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0220] rumbo=289 (orientation)  meta: cp#3 a 26 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0221] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0222] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0223] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0224] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0225] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0226] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0227] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0228] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0229] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0230] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=9% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0231] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0232] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0233] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0234] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0235] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0236] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0237] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0238] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=9% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0239] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0240] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0241] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0242] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0243] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0244] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0245] rumbo=290 (orientation)  meta: cp#3 a 26 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0246] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0247] rumbo=290 (orientation)  meta: cp#3 a 25 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0248] rumbo=292 (orientation)  meta: cp#3 a 25 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0249] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0250] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0251] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0252] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0253] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0254] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=9% cobertura=13%
[bridge]   rumbo -90 grados: libre=4% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0255] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0256] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0257] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0258] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=3% cobertura=43%
[bridge]   rumbo +90 grados: libre=8% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=3%)
[0259] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0260] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0261] rumbo=291 (orientation)  meta: cp#3 a 23 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0262] rumbo=291 (orientation)  meta: cp#3 a 23 m, rel -170 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0263] rumbo=291 (orientation)  meta: cp#3 a 23 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0264] rumbo=291 (orientation)  meta: cp#3 a 23 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0265] rumbo=291 (orientation)  meta: cp#3 a 23 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0266] rumbo=292 (orientation)  meta: cp#3 a 23 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=11% cobertura=13%
[bridge]   rumbo -90 grados: libre=2% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0267] rumbo=292 (orientation)  meta: cp#3 a 23 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0268] rumbo=291 (orientation)  meta: cp#3 a 23 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0269] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0270] rumbo=292 (orientation)  meta: cp#3 a 24 m, rel -169 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0271] rumbo=292 (orientation)  meta: cp#3 a 24 m, rel -168 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0272] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0273] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0274] rumbo=291 (orientation)  meta: cp#3 a 24 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=3% cobertura=43%
[bridge]   rumbo +90 grados: libre=7% cobertura=13%
[bridge]   rumbo -90 grados: libre=4% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=3%)
[0275] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -167 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0276] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -166 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0277] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -166 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0278] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -165 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=9% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0279] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -165 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0280] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -165 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0281] rumbo=291 (orientation)  meta: cp#3 a 25 m, rel -165 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0282] rumbo=291 (orientation)  meta: cp#3 a 26 m, rel -165 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=43%
[bridge]   rumbo +90 grados: libre=10% cobertura=13%
[bridge]   rumbo -90 grados: libre=3% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0283] rumbo=291 (orientation)  meta: cp#3 a 26 m, rel -165 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0284] rumbo=291 (orientation)  meta: cp#3 a 26 m, rel -164 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0285] rumbo=291 (orientation)  meta: cp#3 a 26 m, rel -164 grados  celdas BEV=2994  mapa: 2393 celdas  pose=(+2.22,-0.45,-33gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0286] rumbo=285 (orientation)  meta: cp#3 a 27 m, rel -158 grados  celdas BEV=2994  mapa: 2561 celdas  pose=(+2.22,-0.45,-32gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.30 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=4% cobertura=45%
[bridge]   rumbo +90 grados: libre=11% cobertura=14%
[bridge]   rumbo -90 grados: libre=2% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=4%)
[0287] rumbo=294 (orientation)  meta: cp#3 a 27 m, rel -165 grados  celdas BEV=2994  mapa: 3284 celdas  pose=(+2.20,-0.44,-33gr)
  [ENVIADO] linear=+0.64 angular=+0.45  siguiendo camino (-65 grados)
[0288] rumbo=296 (orientation)  meta: cp#3 a 26 m, rel -166 grados  celdas BEV=2994  mapa: 3409 celdas  pose=(+2.19,-0.44,-33gr)
  [ENVIADO] linear=+0.64 angular=+0.45  siguiendo camino (-65 grados)
[0289] rumbo=315 (orientation)  meta: cp#3 a 27 m, rel +174 grados  celdas BEV=2994  mapa: 3632 celdas  pose=(+2.19,-0.44,-31gr)
[bridge] cambio de lado justificado (+64 grados hacia derecha)
  [ENVIADO] linear=+0.65 angular=-0.45  siguiendo camino (+64 grados)
[0290] rumbo=288 (orientation)  meta: cp#3 a 27 m, rel -159 grados  celdas BEV=2994  mapa: 3727 celdas  pose=(+2.21,-0.45,-31gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0291] rumbo=319 (orientation)  meta: cp#3 a 27 m, rel +169 grados  celdas BEV=2994  mapa: 3819 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0292] rumbo=302 (orientation)  meta: cp#3 a 27 m, rel -176 grados  celdas BEV=2994  mapa: 3819 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0293] rumbo=302 (orientation)  meta: cp#3 a 27 m, rel -176 grados  celdas BEV=2994  mapa: 3819 celdas  pose=(+2.22,-0.46,-34gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.45 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=12% cobertura=64%
[bridge]   rumbo +90 grados: libre=19% cobertura=18%
[bridge]   rumbo -90 grados: libre=28% cobertura=17%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=12%)
[0294] rumbo=302 (orientation)  meta: cp#3 a 27 m, rel -177 grados  celdas BEV=2994  mapa: 3812 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0295] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -178 grados  celdas BEV=2994  mapa: 3802 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0296] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -178 grados  celdas BEV=2994  mapa: 3767 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0297] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -178 grados  celdas BEV=2994  mapa: 3719 celdas  pose=(+2.22,-0.46,-34gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.45 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=19% cobertura=62%
[bridge]   rumbo +90 grados: libre=23% cobertura=17%
[bridge]   rumbo -90 grados: libre=28% cobertura=17%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=19%)
[0298] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -178 grados  celdas BEV=2994  mapa: 3639 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0299] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -178 grados  celdas BEV=2994  mapa: 3551 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0300] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -178 grados  celdas BEV=2994  mapa: 3421 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0301] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -177 grados  celdas BEV=2994  mapa: 3346 celdas  pose=(+2.22,-0.46,-34gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.45 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=32% cobertura=57%
[bridge]   rumbo +90 grados: libre=29% cobertura=17%
[bridge]   rumbo -90 grados: libre=33% cobertura=16%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=32%)
[0302] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -177 grados  celdas BEV=2994  mapa: 3064 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0303] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -177 grados  celdas BEV=2994  mapa: 2410 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0304] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -176 grados  celdas BEV=2994  mapa: 2410 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0305] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -176 grados  celdas BEV=2994  mapa: 2410 celdas  pose=(+2.22,-0.46,-34gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.45 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=9% cobertura=43%
[bridge]   rumbo +90 grados: libre=8% cobertura=13%
[bridge]   rumbo -90 grados: libre=22% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=9%)
[0306] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -175 grados  celdas BEV=2994  mapa: 2410 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0307] rumbo=302 (orientation)  meta: cp#3 a 26 m, rel -175 grados  celdas BEV=2994  mapa: 2410 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0308] rumbo=302 (orientation)  meta: cp#3 a 25 m, rel -174 grados  celdas BEV=2994  mapa: 2378 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0309] rumbo=302 (orientation)  meta: cp#3 a 25 m, rel -173 grados  celdas BEV=2994  mapa: 2378 celdas  pose=(+2.22,-0.46,-34gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.40 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=9% cobertura=42%
[bridge]   rumbo +90 grados: libre=7% cobertura=13%
[bridge]   rumbo -90 grados: libre=21% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=9%)
[0310] rumbo=302 (orientation)  meta: cp#3 a 25 m, rel -173 grados  celdas BEV=2994  mapa: 2378 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0311] rumbo=302 (orientation)  meta: cp#3 a 25 m, rel -172 grados  celdas BEV=2994  mapa: 2378 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
[0312] rumbo=302 (orientation)  meta: cp#3 a 25 m, rel -172 grados  celdas BEV=2994  mapa: 2378 celdas  pose=(+2.22,-0.46,-34gr)
  [ENVIADO] linear=+0.00 angular=+0.00  OBSTACULO al frente
^C
[bridge] parada solicitada, frenando ...
^C
[bridge] parada solicitada, frenando ...
[0313] rumbo=301 (orientation)  meta: cp#3 a 25 m, rel -171 grados  celdas BEV=2994  mapa: 2378 celdas  pose=(+2.22,-0.46,-34gr)
[bridge] REGIMEN CERCANO: 4 frames bloqueado seguidos, clearance=0.45 m
[bridge]   mapa detras del robot: libre=0% cobertura=0%
[bridge]   detras no parece seguro (o sin datos todavia), salteo el retroceso
[bridge]   rumbo +0 grados: libre=9% cobertura=42%
[bridge]   rumbo +90 grados: libre=8% cobertura=13%
[bridge]   rumbo -90 grados: libre=21% cobertura=14%
[bridge]   rumbo +180 grados: libre=0% cobertura=0%
[bridge]   elijo rumbo +0 grados por mapa (libre=9%)
[bridge] frenando el rover

--- resumen ---
  iteraciones:            314
  planes exitosos:        89
  planes vacios:          69
  frenadas por obstaculo: 129
  desatascos forzados:    0
  regimen cercano:        32 (retrocesos: 0)
  recuperaciones:         mapa=49  vlm=0  ciegas=0
  --- memoria espacial ---
  celdas del mapa:        2378
  recentrados:            0
  pose final:             (+2.22, -0.46) -34 grados
  distancia recorrida:    1.89 m
  correcciones GPS:       38
  errores:                0

  --- rumbo: brujula vs GPS ---
  muestras utiles:        0 (pocas para concluir; hace falta que el robot avance lo suficiente como para que gps_track se active)
(.venv) pablolube@D

```